# Spike B — hierarchical-memory (Ch4 Memory)

Three-tier Letta/MemGPT pattern: core (small, fast) + recall (raw history) + archival (overflow, searchable). The durability-aware eviction protects facts like 'production region is us-east-1' while letting short-lived shell commands rotate through.

**DevOps scenario:** a 5-day incident investigation. Day 1 the agent learns durable facts (production AWS account, primary region, on-call rotation). Days 2-5 the agent's working memory floods with short-lived shell tails and log snippets. We then verify the agent can still answer 'what region is production' (durable fact, still in core) AND 'what was the circuit-breaker tail line' (evicted to archival, still searchable).

**Source:** *Agentic Graph RAG* Ch4 — Letta (MemGPT) Approach + Example 4-6.

In [ ]:
import os, sys, json
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SKILL_DIR = os.path.join(REPO_ROOT, "skills", "memory", "hierarchical-memory")
sys.path.insert(0, SKILL_DIR)

from lib import HierarchicalMemory, DURABILITY_DURABLE, DURABILITY_SHORT_LIVED

os.environ.update({
    "AWS_ACCESS_KEY_ID": "testing", "AWS_SECRET_ACCESS_KEY": "testing",
    "AWS_SECURITY_TOKEN": "testing", "AWS_SESSION_TOKEN": "testing",
    "AWS_DEFAULT_REGION": "us-east-1",
})
from moto import mock_aws
import boto3
FICTIONAL_ACCOUNT = "123456789012"
print("setup ok")

## Day 1 — populate core with durable production-environment facts

In [ ]:
mem = HierarchicalMemory(core_limit=10)

for content, dur in [
    (f"Production AWS account is {FICTIONAL_ACCOUNT}", DURABILITY_DURABLE),
    ("Primary region is us-east-1", DURABILITY_DURABLE),
    ("On-call rotation: Sarah (primary), Marcus (secondary)", DURABILITY_DURABLE),
    ("Checkout API uses synchronous HTTP to payments service", DURABILITY_DURABLE),
    ("Service-checkout-api runs EC2 m5.xlarge", DURABILITY_DURABLE),
]:
    mem.add_fact(content, dur)

print(f"After Day 1: core={len(mem.core)}, recall={len(mem.recall)}, archival={len(mem.archival)}")
for f in mem.core.values():
    print(f"  [{f.durability}] {f.content}")

## Days 2-5 — flood with short-lived shell tails and log snippets

In [ ]:
short_lived_pool = [
    "Running: aws ec2 describe-instances", "Tail: 08:00 504 gateway",
    "Running: kubectl get pods", "Tail: 08:01 503",
    "Running: gh pr view 1234", "Tail: 08:02 retry-storm",
    "Running: terraform plan", "Tail: 08:03 circuit-breaker tripped",
    "Running: psql -c 'SELECT count(*)'", "Tail: 08:04 db-pool full",
    "Running: redis-cli ping", "Tail: 08:05 p99 8200ms",
    "Running: aws sts get-caller-identity", "Tail: 08:06 page-fired",
]
for s in short_lived_pool:
    mem.add_fact(s, DURABILITY_SHORT_LIVED)

print(f"After flood: core={len(mem.core)}, recall={len(mem.recall)}, archival={len(mem.archival)}")

## Verify: durable facts survived, short-lived facts evicted, archival still queryable

In [ ]:
# Q1: 'what region is production?' — durable, should still be in core
result_region = mem.query("region")
print("Q1: region")
print(json.dumps(result_region, indent=2))
core_has_region = any("us-east-1" in r["content"] for r in result_region["core"])
assert core_has_region, "durable region fact should still be in core after flood"
print("\n[PASS] durable 'us-east-1' fact survived eviction")

In [ ]:
# Q2: 'circuit-breaker' — short-lived, evicted to archival, still queryable
result_cb = mem.query("circuit-breaker")
print("Q2: circuit-breaker")
print(json.dumps(result_cb, indent=2))
archival_has_cb = any("circuit-breaker" in r["content"] for r in result_cb["archival"])
assert archival_has_cb, "circuit-breaker tail should be findable in archival after eviction"
print("\n[PASS] evicted 'circuit-breaker' fact still queryable from archival")

## Cross-check against moto-mocked AWS: production region matches the durable fact

In [ ]:
# The agent's core memory says us-east-1. We verify by checking that an EC2 client
# in us-east-1 can list the resources we'd expect for the production account.
with mock_aws():
    region_fact = next(r for r in mem.query("region")["core"] if "us-east-1" in r["content"])
    durable_region = "us-east-1"
    ec2 = boto3.client("ec2", region_name=durable_region)
    # Spin up an instance in the durable region to confirm shape
    images = ec2.describe_images()["Images"]
    ami = images[0]["ImageId"] if images else "ami-12345678"
    run = ec2.run_instances(ImageId=ami, MinCount=1, MaxCount=1, InstanceType="m5.xlarge")
    inst = run["Instances"][0]
    inst_az = inst["Placement"]["AvailabilityZone"]
    assert inst_az.startswith(durable_region), f"AZ {inst_az} not in durable region {durable_region}"
    print(f"durable region from core memory: {durable_region}")
    print(f"moto EC2 instance AZ:            {inst_az}")
    print("\n[PASS] Durable region fact matches moto-AWS instance AZ.")

## Diagnostics — health check should be green

In [ ]:
diag = mem.diagnostics()
print(json.dumps(diag, indent=2))
# 5 durable + (10 - 5) = 5 short-lived in core; no pathological flag
# (the durability_protection_factor protects durables, so the 'short>50%' warning may or may not fire)

## Conclusion
- Durable production-environment facts survived a 14-item short-lived flood (5 evictions targeted the short-lived facts).
- Evicted facts are still findable in archival — not deleted.
- The durable region fact matches what `moto`-mocked AWS reports as the EC2 instance's AZ region prefix.

**The three-tier hierarchy made forgetting an explicit design choice rather than an accident.**